In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('student_grade_ses_dataset.csv')
df.head()

,student_id,academic_year,survey_date,age,gender,grade_level,school_type,residence_type,family_size,num_siblings,...,distance_to_school_km,scholarship_status,extra_tutoring,parental_involvement_score,study_hours_per_week,attendance_rate_pct,previous_year_grade_pct,final_grade_pct,grade_class,at_risk_flag
0,STU101529,2024-2025,6/3/25,16,Male,12,Government,Rural,3,0,...,4.0,No Scholarship,0,4,15.0,76.8,61.7,59.4,D,0
1,STU107546,2022-2023,2/5/25,15,Male,10,Government,Urban,3,1,...,0.7,No Scholarship,0,3,20.9,74.5,49.2,60.2,C,0
2,STU103737,2024-2025,21/3/25,14,Male,9,Government,Urban,2,0,...,2.9,No Scholarship,0,4,20.9,94.9,56.9,58.4,D,0
3,STU108172,2023-2024,30/3/25,13,Male,8,Government,Rural,6,4,...,7.0,No Scholarship,0,1,6.5,63.6,49.3,57.0,D,0
4,STU110633,2023-2024,6/5/25,11,Female,6,Private-Unaided,Rural,5,3,...,9.1,No Scholarship,0,5,13.9,86.0,79.3,88.7,B,0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12025 entries, 0 to 12024
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  12025 non-null  str    
 1   academic_year               12025 non-null  str    
 2   survey_date                 12025 non-null  str    
 3   age                         12025 non-null  int64  
 4   gender                      12025 non-null  str    
 5   grade_level                 12025 non-null  int64  
 6   school_type                 12025 non-null  str    
 7   residence_type              12025 non-null  str    
 8   family_size                 12025 non-null  int64  
 9   num_siblings                12025 non-null  int64  
 10  single_parent_household     12025 non-null  int64  
 11  family_income_bracket       12025 non-null  str    
 12  annual_family_income_inr    11304 non-null  float64
 13  father_education_level      11605 non-null

## Drop duplicates

In [3]:
# Confirm duplicated student_ids are duplicate rows

# 1. Rows that share the same 'id'
id_dups = df[df.duplicated(subset=['student_id'], keep=False)]

# 2. Rows that share the same 'id' AND are identical across ALL columns
full_dups = df[df.duplicated(subset=['student_id'], keep=False) & df.duplicated(keep=False)]

print(f"Total rows with duplicate IDs: {len(id_dups)}")
print(f"Rows with duplicate IDs that are FULLY identical: {len(full_dups)}")
print(f"Rows with duplicate IDs that have DIFFERENCES: {len(id_dups) - len(full_dups)}")

df = df.drop_duplicates(subset=['student_id']) 

Total rows with duplicate IDs: 50
Rows with duplicate IDs that are FULLY identical: 50
Rows with duplicate IDs that have DIFFERENCES: 0


## Missing values

In [7]:
null_summary = pd.DataFrame({ "nulls": df.isna().sum() })
null_summary[null_summary["nulls"] > 0]

,nulls
annual_family_income_inr,720
father_education_level,420
mother_education_level,360
father_occupation,480
mother_occupation,600
has_internet_access,180
distance_to_school_km,240
attendance_rate_pct,300
previous_year_grade_pct,360


## Clean

Parent education / occupation NULLs look random, so they become their own "Unknown" category. This learns nothing from the data, so its safe to do before CV.

In [9]:
parent_cols = ['father_education_level', 'mother_education_level', 'father_occupation', 'mother_occupation']

df[parent_cols] = df[parent_cols].fillna("Unknown")

nulls = df.isna().sum()

nulls[nulls > 0]


annual_family_income_inr    720
has_internet_access         180
distance_to_school_km       240
attendance_rate_pct         300
previous_year_grade_pct     360
dtype: int64

## Filling the rest

Fills values have to come from training rows only.
The `fill_missing(train, test)` function does a fit and transform inside each CV fold.

| Column | Fill with | Why |
|---|---|---|
| `annual_family_income_inr` | median per `family_income_bracket`| bracket is complete and tracks income closely (Low ≈ 103k vs High ≈ 3M); the overall median (~350k) would put a high-income family in the middle |
| `distance_to_school_km` | median per `residence_type` | Rural ≈ 5.4 km vs Urban ≈ 1.7 km |
| `has_internet_access` | most common value per `residence_type`| Rural 36% vs Urban 60%; the overall mode would give everyone 0 |
| `attendance_rate_pct`, `previous_year_grade_pct` | overall median | nothing else predicts them well (correlation ≤ 0.29); |

Rows with these missing values have about the same final grade as complete rows, so the gaps look random and simple filling is reasonable. `family_income_bracket` is dropped once income is filled.

In [42]:
def fill_missing(train, test):
    train = train.copy()
    test = test.copy()

    # income -> median of income bracket
    income = "annual_family_income_inr"
    bracket = "family_income_bracket"
    medians = train.groupby(bracket)[income].median()
    train_fill = train[bracket].map(medians)
    test_fill = test[bracket].map(medians)
    train[income] = train[income].fillna(train_fill)
    test[income] = test[income].fillna(test_fill)

    # distance -> median of residence type
    distance = "distance_to_school_km"
    residence = "residence_type"
    medians = train.groupby(residence)[distance].median()
    train_fill = train[residence].map(medians)
    test_fill = test[residence].map(medians)
    train[distance] = train[distance].fillna(train_fill)
    test[distance] = test[distance].fillna(test_fill)

    # internet -> most common value for residence type
    internet = 'has_internet_access'
    most_common = train.groupby(residence)[internet].mean().round()
    train_fill = train[residence].map(most_common)
    test_fill = test[residence].map(most_common)
    train[internet] = train[internet].fillna(train_fill)
    test[internet] = test[internet].fillna(test_fill)

    # attendance and previous grade -> overall median
    other_cols = ['attendance_rate_pct', 'previous_year_grade_pct']
    medians = train[other_cols].median()
    train = train.fillna(medians)
    test = test.fillna(medians)

    # we can drop the bucket column after this
    train = train.drop(columns=bracket)
    test = test.drop(columns=bracket)

    return train, test


## Target and predictors

In [37]:
TARGET_COL = 'final_grade_pct'

# # Heavily skewed income, so log-transform it.
# df['log_income'] = np.log(df['annual_family_income_inr'])

CONTINUOUS_COLS = [
    'distance_to_school_km', 'study_hours_per_week', 'attendance_rate_pct',
    'previous_year_grade_pct', 'annual_family_income_inr',
]

# Integer-coded predictors; term types are specified in the GAM section.
DISCRETE_COLS = [
    'age', 'grade_level', 'family_size', 'num_siblings',
    'single_parent_household', 'has_internet_access',
    'has_computer_or_laptop', 'extra_tutoring', 'parental_involvement_score',
]

# Match the Project.ipynb predictor set.
CATEGORICAL_COLS = [
    'academic_year', 'gender', 'school_type', 'residence_type',
    'family_income_bracket', 'father_education_level', 'mother_education_level',
    'father_occupation', 'mother_occupation', 'scholarship_status',
]
NUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS
PREDICTOR_COLS = NUMERIC_COLS + CATEGORICAL_COLS

In [38]:
X = df[PREDICTOR_COLS]
Y = df[TARGET_COL]
print('X:', X.shape, ' Y:', Y.shape)

# Match Project.ipynb: stratified 85/15 split; use training data for CV only.
from sklearn.model_selection import train_test_split
TEST_SIZE = 0.15
SEED = 42
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, stratify=df['grade_class'], random_state=SEED
)
print('train:', X_train.shape, ' test:', X_test.shape)

X: (12000, 24)  Y: (12000,)
train: (10200, 24)  test: (1800, 24)


In [43]:
X_train, X_test = fill_missing(X_train, X_test)

In [48]:
X_train.isna().sum().sum() + X_test.isna().sum().sum()

np.int64(0)